In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from tqdm.notebook import tqdm

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")

# Input files
ALL_TWEETS_AGGREGATED = "C:/Users/skazempour/Documents/StockTwits/dataset/v1/data/aggregated/aggregated_sentiment_returns.pkl"
LINEAR_REGRESSION_PREDICTIONS = os.path.join(DATA, "predictions_linear_regression.pkl")
DECISION_TREE_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree.pkl")
DECISION_TREE_SHALLOW_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_shallow.pkl")
DECISION_TREE_MODERATE_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_moderate.pkl")
DECISION_TREE_CONSTRAINED_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_constrained.pkl")
DECISION_TREE_CV_PRUNED_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_cv_pruned.pkl")
RANDOM_FOREST_PREDICTIONS = os.path.join(DATA, "predictions_random_forest.pkl")
NEURAL_NETWORK_PREDICTIONS = os.path.join(DATA, "predictions_neural_network.pkl")

# Put together all predictions

In [3]:
# Load the original aggregated tweets data
tweets_aggregated = pd.read_pickle(ALL_TWEETS_AGGREGATED)
tweets_aggregated = tweets_aggregated[tweets_aggregated['date'] >= '2012-01-01']

# Add the linear regression model
linear_regression_predictions = pd.read_pickle(LINEAR_REGRESSION_PREDICTIONS).drop(columns=['index'])
linear_regression_predictions.columns = ['date', 'symbol', 'lr_exp', 'lr_roll_252', 'lr_roll_21']
df = pd.merge(tweets_aggregated, linear_regression_predictions, on=['date', 'symbol']) # Clean merge

# Add the decision tree model without prunning
decision_tree_predictions = pd.read_pickle(DECISION_TREE_PREDICTIONS).drop(columns=['index'])
decision_tree_predictions.columns = ['date', 'symbol', 'dt_exp', 'dt_roll_252', 'dt_roll_21']
df = pd.merge(df, decision_tree_predictions, on=['date', 'symbol']) # Clean merge

# Add the shallow decision tree model
decision_tree_shallow_predictions = pd.read_pickle(DECISION_TREE_SHALLOW_PREDICTIONS).drop(columns=['index'])
decision_tree_shallow_predictions.columns = ['date', 'symbol', 'dt_shallow_exp', 'dt_shallow_roll_252', 'dt_shallow_roll_21']
df = pd.merge(df, decision_tree_shallow_predictions, on=['date', 'symbol']) # Clean merge

# Add the moderate decision tree model
decision_tree_moderate_predictions = pd.read_pickle(DECISION_TREE_MODERATE_PREDICTIONS).drop(columns=['index'])
decision_tree_moderate_predictions.columns = ['date', 'symbol', 'dt_moderate_exp', 'dt_moderate_roll_252', 'dt_moderate_roll_21']
df = pd.merge(df, decision_tree_moderate_predictions, on=['date', 'symbol']) # Clean merge

# Add the constrained decision tree model
decision_tree_constrained_predictions = pd.read_pickle(DECISION_TREE_CONSTRAINED_PREDICTIONS).drop(columns=['index'])
decision_tree_constrained_predictions.columns = ['date', 'symbol', 'dt_constrained_exp', 'dt_constrained_roll_252', 'dt_constrained_roll_21']
df = pd.merge(df, decision_tree_constrained_predictions, on=['date', 'symbol']) # Clean merge

# Add the cv pruned decision tree model
decision_tree_cv_pruned_predictions = pd.read_pickle(DECISION_TREE_CV_PRUNED_PREDICTIONS).drop(columns=['index'])
decision_tree_cv_pruned_predictions.columns = ['date', 'symbol', 'dt_cv_pruned_exp', 'dt_cv_pruned_roll_252', 'dt_cv_pruned_roll_21']
df = pd.merge(df, decision_tree_cv_pruned_predictions, on=['date', 'symbol']) # Clean merge

# Add the random forest model
random_forest_predictions = pd.read_pickle(RANDOM_FOREST_PREDICTIONS).drop(columns=['index'])
random_forest_predictions.columns = ['date', 'symbol', 'rf_exp', 'rf_roll_252', 'rf_roll_21']
df = pd.merge(df, random_forest_predictions, on=['date', 'symbol']) # Clean merge

# Add the neural network model
neural_network_predictions = pd.read_pickle(NEURAL_NETWORK_PREDICTIONS).drop(columns=['index'])
neural_network_predictions.columns = ['date', 'symbol', 'nn_exp', 'nn_roll_252', 'nn_roll_21']
df = pd.merge(df, neural_network_predictions, on=['date', 'symbol']) # Clean merge


# Run sentiment regressions

In [4]:
prediction_columns = ['lr_exp', 'lr_roll_252', 'lr_roll_21',
                      'dt_exp', 'dt_roll_252', 'dt_roll_21',
                      'dt_shallow_exp', 'dt_shallow_roll_252', 'dt_shallow_roll_21',
                      'dt_moderate_exp', 'dt_moderate_roll_252', 'dt_moderate_roll_21',
                      'dt_constrained_exp', 'dt_constrained_roll_252', 'dt_constrained_roll_21',
                      'dt_cv_pruned_exp', 'dt_cv_pruned_roll_252', 'dt_cv_pruned_roll_21',
                      'rf_exp', 'rf_roll_252', 'rf_roll_21',
                      'nn_exp', 'nn_roll_252', 'nn_roll_21']

ret_col = 'ar_FF5_1'
reg_results = []
for pred_col in tqdm(prediction_columns):
    reg_data = df[['permno','date', ret_col, pred_col]].dropna().copy()
    reg_data['date'] = pd.to_datetime(reg_data['date'])
    reg_data['date'] = reg_data['date'].dt.year * 10000 + reg_data['date'].dt.month*100 + reg_data['date'].dt.day 
    reg_data = reg_data.rename(columns={pred_col: 'pred'})
    y = reg_data[ret_col]
    X = sm.add_constant(reg_data['pred'])
    model = sm.OLS(y, X, missing='drop').fit(cov_type='cluster',cov_kwds={'groups':np.array(reg_data[['permno','date']])})
    reg_results.append(model)

  0%|          | 0/24 [00:00<?, ?it/s]

# Render the results in a table

In [5]:
from latex_table import linear_regression

# Format and save the table
vars_to_include = ['pred', 'const']
var_names = ["Prediction", "Const."]
rename_dict = dict(zip(vars_to_include, var_names))

tbl = linear_regression(reg_results)
tbl.rename_variables(rename_dict)
tbl.columns = prediction_columns
tbl.obs = True
tbl.float_format = ".2f"
tbl.render(midrule=True)
tbl.tbl


lr_exp    lr_roll_252     lr_roll_21     dt_exp  \
Prediction    0.91THREESTAR  0.86THREESTAR  0.48THREESTAR       0.00   
                     (0.10)         (0.10)         (0.07)     (0.01)   
Const.          0.00TWOSTAR          -0.00          -0.00      -0.00   
                     (0.00)         (0.00)         (0.00)     (0.00)   
                                                                       
N                 3,001,267      3,001,267      3,001,267  3,001,267   

             dt_roll_252 dt_roll_21 dt_shallow_exp dt_shallow_roll_252  \
Prediction          0.01       0.00           0.01                0.03   
                  (0.00)     (0.00)         (0.07)              (0.05)   
Const.             -0.00      -0.00          -0.00               -0.00   
                  (0.00)     (0.00)         (0.00)              (0.00)   
                                                                         
N              3,001,267  3,001,267      3,001,267           3,001,267   

             dt_shallow_roll_21 dt_moderate_exp  ... dt_constrained_roll_21  \
Prediction          0.03TWOSTAR   0.14THREESTAR  ...          0.03THREESTAR   
                         (0.01)          (0.03)  ...                 (0.01)   
Const.                    -0.00            0.00  ...                  -0.00   
                         (0.00)          (0.00)  ...                 (0.00)   
                                                 ...                          
N                     3,001,267       3,001,267  ...              3,001,267   

             dt_cv_pruned_exp dt_cv_pruned_roll_252 dt_cv_pruned_roll_21  \
Prediction               0.00                 -0.00          0.01ONESTAR   
                       (0.02)                (0.01)               (0.01)   
Const.                  -0.00                 -0.00                -0.00   
                       (0.00)                (0.00)               (0.00)   
                                                                           
N                   3,001,267             3,001,267            3,001,267   

                     rf_exp    rf_roll_252     rf_roll_21         nn_exp  \
Prediction    0.25THREESTAR  0.22THREESTAR  0.09THREESTAR  0.36THREESTAR   
                     (0.05)         (0.04)         (0.02)         (0.07)   
Const.                 0.00          -0.00          -0.00           0.00   
                     (0.00)         (0.00)         (0.00)         (0.00)   
                                                                           
N                 3,001,267      3,001,267      3,001,267      3,001,267   

                nn_roll_252      nn_roll_21  
Prediction    0.26THREESTAR   0.11THREESTAR  
                     (0.04)          (0.02)  
Const.                -0.00           -0.00  
                     (0.00)          (0.00)  
                             ADDMIDRULEHERE  
N                 3,001,267       3,001,267  

[6 rows x 24 columns]